# ARTI 402 — Deep Learning
## Lab 8 — Submission (PyTorch: Writing the Training Loop Again, On Purpose)

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch :", torch.__version__)

def spiral_data(samples, classes, seed=0):
    rng = np.random.RandomState(seed)
    X = np.zeros((samples * classes, 2))
    y = np.zeros(samples * classes, dtype="uint8")
    for c in range(classes):
        ix = range(samples * c, samples * (c + 1))
        r = np.linspace(0.0, 1, samples)
        t = np.linspace(c * 4, (c + 1) * 4, samples) + rng.randn(samples) * 0.2
        X[ix] = np.c_[r * np.sin(t * 2.5), r * np.cos(t * 2.5)]
        y[ix] = c
    return X, y

X_train, y_train = spiral_data(samples=40,  classes=3, seed=0)
X_val,   y_val   = spiral_data(samples=100, classes=3, seed=2)

print("\ntrain:", X_train.shape, " validation:", X_val.shape)
print("\nSetup OK")

PyTorch : 2.11.0+cpu

train: (120, 2)  validation: (300, 2)

Setup OK


In [3]:
from arti402_figures import draw_overfitting_curves, draw_decision_boundaries
print("figure helpers loaded")

figure helpers loaded


In [4]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)

loss = (w * x - 5.0) ** 2
loss.backward()

print("loss      :", loss.item())
print("w.grad    :", w.grad.item())
print("by hand   :", 2 * (2 * 3 - 5) * 3, "   <- dL/dw = 2(wx - 5)x")

loss      : 1.0
w.grad    : 6.0
by hand   : 6    <- dL/dw = 2(wx - 5)x


In [5]:
w = torch.tensor(2.0, requires_grad=True)

for step in range(3):
    loss = (w * 3 - 5.0) ** 2
    loss.backward()
    print(f"backward #{step + 1}:  w.grad = {w.grad.item():.1f}")

print("\nThe same gradient, added three times.")

backward #1:  w.grad = 6.0
backward #2:  w.grad = 12.0
backward #3:  w.grad = 18.0

The same gradient, added three times.


In [6]:
### Exercise 1 — check that autograd is telling the truth
torch.manual_seed(0)
layer = nn.Linear(2, 3).double()
Xg = torch.randn(5, 2, dtype=torch.float64)
yg = torch.tensor([0, 1, 2, 0, 1])

def loss_of(model):
    return F.cross_entropy(model(Xg), yg)

loss = loss_of(layer)
loss.backward()
auto = layer.weight.grad.clone()

numeric = torch.zeros_like(auto)
h = 1e-6
with torch.no_grad():
    for i in range(auto.shape[0]):
        for j in range(auto.shape[1]):
            original = layer.weight[i, j].item()

            layer.weight[i, j] = original + h
            L_plus = loss_of(layer).item()

            layer.weight[i, j] = original - h
            L_minus = loss_of(layer).item()

            layer.weight[i, j] = original
            numeric[i, j] = (L_plus - L_minus) / (2 * h)

difference = (auto - numeric).abs().max().item()
print(f"max |autograd - numerical| = {difference:.2e}")

assert difference < 1e-8
print("\nAutograd agrees with the definition of a derivative.")

max |autograd - numerical| = 5.93e-11

Autograd agrees with the definition of a derivative.


### Checkpoint 1
1. `loss.backward()` filled in `layer.weight.grad`. Which rule from Lab 2 is PyTorch applying, over and over, to work that out?
2. What would go wrong if a training loop called `backward()` and `step()` but never `zero_grad()`?
3. The numerical gradient needed two forward passes **per weight**. Our layer has 6 weights, so 12 passes. A network with a million weights would need two million. In one sentence, why is autograd not simply a faster version of the same idea?

1. The **Chain Rule**.
2. The gradients would **accumulate (add together)** across every batch instead of resetting, causing the optimizer to take massively incorrect steps based on the sum of all previous gradients.
3. Autograd computes the exact analytical gradient for all weights simultaneously in a single backward pass, rather than approximating them one by one through millions of individual forward passes.

In [7]:
### Exercise 2 — build the Lab 6 network
class Net(nn.Module):
    def __init__(self, hidden=128, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.fc2 = nn.Linear(hidden, 3)

    def forward(self, x):
        return self.fc2(self.drop(self.relu(self.fc1(x))))

torch.manual_seed(0)
model = Net()
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\ntotal parameters: {n_params}")
for name, p in model.named_parameters():
    print(f"   {name:12} {str(tuple(p.shape)):10} {p.numel():>5}")

assert n_params == 771
assert model(torch.zeros(4, 2)).shape == (4, 3)
print("\nExercise 2 passed")

Net(
  (fc1): Linear(in_features=2, out_features=128, bias=True)
  (relu): ReLU()
  (drop): Identity()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

total parameters: 771
   fc1.weight   (128, 2)     256
   fc1.bias     (128,)       128
   fc2.weight   (3, 128)     384
   fc2.bias     (3,)           3

Exercise 2 passed


In [8]:
### Exercise 3 — write it
def train_one_epoch(model, optimizer, X, y, batch_size=32):
    model.train()
    n = len(X)
    perm = torch.randperm(n)
    total = 0.0

    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        xb, yb = X[idx], y[idx]

        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()

        total += loss.item() * len(idx)

    return total / n

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xva = torch.tensor(X_val,   dtype=torch.float32)
yva = torch.tensor(y_val,   dtype=torch.long)

torch.manual_seed(0)
m = Net()
opt = torch.optim.Adam(m.parameters(), lr=0.02)
first = train_one_epoch(m, opt, Xtr, ytr)
for _ in range(9):
    last = train_one_epoch(m, opt, Xtr, ytr)

print(f"epoch  1 loss: {first:.3f}")
print(f"epoch 10 loss: {last:.3f}")
assert first < 1.5
assert last < first - 0.05
print("\nIt learns. You wrote the loop.")

epoch  1 loss: 1.201
epoch 10 loss: 1.003

It learns. You wrote the loop.


In [9]:
### Exercise 4 — evaluating properly
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = F.cross_entropy(logits, y).item()
        accuracy = (logits.argmax(dim=1) == y).float().mean().item()
    return loss, accuracy

loss_v, acc_v = evaluate(m, Xva, yva)
print(f"validation loss {loss_v:.3f}   accuracy {acc_v:.3f}")
assert 0.0 <= acc_v <= 1.0
assert not m.training
print("\nExercise 4 passed")

validation loss 1.053   accuracy 0.417

Exercise 4 passed


### Checkpoint 2
1. Put the four training-step lines in the correct order, and say what breaks if `zero_grad()` is moved to **after** `step()`.
2. `model.eval()` and `torch.no_grad()` are often written together but do completely different jobs. What does each one do?
3. Your model's `forward` has no softmax, yet `evaluate` computes accuracy with `argmax`. Why is the answer the same with or without a softmax?

1. Order: 1. `optimizer.zero_grad()`, 2. `loss = F.cross_entropy(...)`, 3. `loss.backward()`, 4. `optimizer.step()`. (Note: Moving `zero_grad()` to *after* `step()` actually works fine for the next iteration, but if you mistakenly place it between `backward()` and `step()`, you will delete the gradients before the optimizer can use them, meaning the network won't learn anything).
2. `model.eval()` changes the **behavior of layers** (e.g., turns off Dropout and uses population statistics for Batch Norm). `torch.no_grad()` turns off the **autograd engine**, saving memory and compute by not building the computational graph.
3. Softmax is a **monotonic function**, meaning it strictly preserves the order of values. The class with the largest raw score (logit) will always become the largest probability, so `argmax` gives the exact same predicted class either way.

In [10]:
### Exercise 5 — the same loop, with a DataLoader
def train_one_epoch_loader(model, optimizer, loader):
    model.train()
    total, seen = 0.0, 0

    for xb, yb in loader:
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()

        total += loss.item() * len(xb)
        seen += len(xb)

    return total / seen

torch.manual_seed(0)
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True,
                    generator=torch.Generator().manual_seed(0))

m2 = Net()
opt2 = torch.optim.Adam(m2.parameters(), lr=0.02)
for _ in range(10):
    final = train_one_epoch_loader(m2, opt2, loader)

print(f"after 10 epochs, loss {final:.3f}")
print(f"batches per epoch: {len(loader)}   (120 samples, batch size 32)")

assert len(loader) == 4
assert final < 1.0
print("\nExercise 5 passed")

after 10 epochs, loss 0.996
batches per epoch: 4   (120 samples, batch size 32)

Exercise 5 passed


In [11]:
torch.manual_seed(0)
drop = nn.Dropout(0.5)
x = torch.ones(1, 10)

drop.train()
print("train mode:", drop(x).numpy().round(1), "  <- half zeroed, rest doubled to keep the scale")
drop.eval()
print("eval  mode:", drop(x).numpy().round(1), "  <- untouched")

train mode: [[0. 0. 2. 0. 0. 0. 2. 2. 0. 2.]]   <- half zeroed, rest doubled to keep the scale
eval  mode: [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]   <- untouched


In [12]:
### Exercise 6 — a full training run, with regularization
def run(hidden=128, dropout=0.0, weight_decay=0.0, epochs=1500,
        batch_size=32, lr=0.02, seed=0, record_every=5):
    torch.manual_seed(seed)
    model = Net(hidden, dropout)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))

    history = []
    for epoch in range(epochs):
        train_one_epoch_loader(model, optimizer, loader)

        if epoch % record_every == 0 or epoch == epochs - 1:
            tr_loss, tr_acc = evaluate(model, Xtr, ytr)
            va_loss, va_acc = evaluate(model, Xva, yva)
            history.append((epoch, tr_loss, tr_acc, va_loss, va_acc))

    return model, history

t0 = time.time()
_m, _h = run(epochs=100)
print(f"100 epochs in {time.time() - t0:.1f}s, {len(_h)} rows recorded")
assert len(_h) == 21
assert _h[-1][2] > _h[0][2]
print("ready for the assessment")

100 epochs in 0.4s, 21 rows recorded
ready for the assessment
